# 11 language-script group dataset for continued fine-tuning

Builds the dataset for the continued fine-tuning experiment across exactly
**11 language-script groups**.

| Group | label (ISO-3) | script | Source |
| --- | --- | --- | --- |
| `Sinhala-Sinh` | `sin` | `Sinh` | existing target `train.csv` / `val.csv` |
| `Pali-Sinh` | `pli` | `Sinh` | existing target `train.csv` / `val.csv` |
| `Sanskrit-Sinh` | `san` | `Sinh` | existing target `train.csv` / `val.csv` |
| `Sanskrit-Deva` | `san` | `Deva` | `surajp/sanskrit_classic` |
| `English-Latn` | `eng` | `Latn` | `CohereLabs/aya_dataset` |
| `Tamil-Taml` | `tam` | `Taml` | `CohereLabs/aya_dataset` |
| `Hindi-Deva` | `hin` | `Deva` | `CohereLabs/aya_dataset` |
| `Bengali-Beng` | `ben` | `Beng` | `CohereLabs/aya_dataset` |
| `Arabic-Arab` | `arb` | `Arab` | `CohereLabs/aya_dataset` |
| `French-Latn` | `fra` | `Latn` | `CohereLabs/aya_dataset` |
| `German-Latn` | `deu` | `Latn` | `CohereLabs/aya_dataset` |

All other Aya languages have been removed from this configuration.

Both Sanskrit groups keep `label = "san"`; `script` and `group` distinguish
them. Labels are **not** converted to fastText output IDs here.

## Outputs (new names -- earlier files are preserved)

| File | Contents |
| --- | --- |
| `datasets/finetuning/train_mixed_11groups.jsonl` | Training records, all 11 groups |
| `datasets/finetuning/val_mixed_11groups.jsonl` | Validation records, all 11 groups |
| `datasets/finetuning/dataset_11groups_report.json` | Seed, sources, exclusions, validation findings |

The older `train_mixed.jsonl` / `val_mixed.jsonl` from the previous
22-language rehearsal experiment are **left untouched**.

## Record schema

Each line carries `text`, `label` (ISO-3), `script`, `group`, `source` and
`provenance`, plus whatever original identifiers the source provided
(`orig_id`, `subcorpus`, `group_id`, `aya_row_index`, `line_index`).

## Key properties

- **Target splits preserved.** Sinhala, Pali and Sanskrit-Sinh keep their
  existing train/val assignment verbatim; those files are never recombined
  and randomly re-split.
- **Reproducible sampling.** Seed 42, `random.sample` over the deduplicated
  eligible pool -- not the first N rows. Cap of 5,000 per language,
  configurable via `MAX_PER_LANG`. Sources with fewer eligible examples keep
  everything available; no row is ever duplicated to reach the quota.
- **Grouped 90/10 split** on the newly sampled sources, keyed on the
  NFC-normalized text so identical texts never straddle train and val.
- **Script checked separately from the label.** Metadata alone is not taken
  as proof; mixed-script and low-purity examples are reported and excluded.
- **Benchmarks stay separate.** FLORES+, CommonLID and WiLI-2018 are read
  only for overlap reporting -- never for training, validation, early
  stopping or model selection, and never modified.


In [ ]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas datasets huggingface_hub')
    print("Setup complete!")


## Sanskrit-Devanagari source

`CohereLabs/aya_dataset` contains **no Sanskrit at all** (verified: `san`
appears in 0 of its 70 language codes), so this group needs an independent
source.

We reuse the source already documented for this class in
`docs/DATA_DICTIONARY_11LANG_HYBRID.md`: **`surajp/sanskrit_classic`**. That
HF repo ships only a legacy loader script, so we fetch its upstream data
archive directly and record the provenance (URL + SHA-256) in the report.

This corpus has no predefined splits, so the same cap and 90/10 grouped
split used for the Aya languages is applied to it.

We do **not** transliterate Sinhala-script Sanskrit and relabel it as
Devanagari, and we do **not** take Sanskrit from FLORES+/CommonLID/WiLI.


In [ ]:
import os
import json
import pandas as pd
from datasets import load_dataset
import random

if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

output_dir = 'datasets/finetuning'
train_jsonl_path = os.path.join(output_dir, "train.jsonl")
output_mixed_path = os.path.join(output_dir, "train_mixed.jsonl")

# Target languages to preserve (excluding Sinhala, Pali, Sanskrit)
TARGET_LANGUAGES = {
    "eng": "en", "hin": "hi", "arb": "ar", "fra": "fr", "deu": "de", "ben": "bn", "tam": "ta",
    "jpn": "ja", "nld": "nl", "pol": "pl", "ita": "it", "por": "pt", "tur": "tr", "spa": "es",
    "ell": "el", "urd": "ur", "zho": "zh", "rus": "ru", "tha": "th", "swh": "sw", "vie": "vi"
}

SAMPLES_PER_LANG = 5000


In [ ]:
# Build the 11-group dataset.
import importlib.util, sys

print("Filtering for target rehearsal languages...")
rehearsal_records = []
lang_counts = {lang: 0 for lang in TARGET_LANGUAGES.keys()}

for row in aya_dataset:
    lang = row['language_code']
    if lang in TARGET_LANGUAGES and lang_counts[lang] < SAMPLES_PER_LANG:
        rehearsal_records.append({
            "text": row["inputs"],
            "label": lang,
            "source": "aya_dataset"
        })
        lang_counts[lang] += 1

    # Stop early if all quotas are met
    if all(count == SAMPLES_PER_LANG for count in lang_counts.values()):
        break

print(f"Sampled rehearsal distributions from Aya Dataset:")
for lang, count in lang_counts.items():
    print(f"  {lang}: {count}")


In [ ]:
print(f"\nLoading original training data from {train_jsonl_path}...")
train_records = []
if os.path.exists(train_jsonl_path):
    with open(train_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            train_records.append(json.loads(line))
    print(f"Loaded {len(train_records)} original training records (Sinhala, Pali, Sanskrit).")
else:
    print(f"WARNING: {train_jsonl_path} not found. Please run setup_finetune_data.ipynb first.")

print("\nCombining and shuffling datasets...")
import random
mixed_records = train_records + rehearsal_records
random.shuffle(mixed_records)

with open(output_mixed_path, 'w', encoding='utf-8') as f:
    for record in mixed_records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"Created {output_mixed_path} with {len(mixed_records)} total mixed records.")
